# Amatrice Paper Experiment Guide (TCN + Histogram Matching)

This notebook implements the full experiment workflow for the Amatrice paper using:

- **TCN (Temporal Convolutional Network)** as the sequence model
- **Histogram matching** preprocessing to remove systematic acquisition biases
- **NDI** and **log-NDI** scoring modes

| Run name | Feature | Model | Preprocessing | Scoring |
|---|---|---|---|---|
| `filt_fine_std_difference` | phase-STD | — (pre/post difference) | — | NDI |
| `filt_fine_cor_difference` | coherence | — (pre/post difference) | — | NDI |
| `tcn_notime_filt_fine_std_ndi` | phase-STD | TCN, no time | histogram match | NDI |
| `tcn_time_filt_fine_std_ndi` | phase-STD | TCN + time | histogram match | NDI |
| `tcn_time_filt_fine_std_logndi` | phase-STD | TCN + time | histogram match | log-NDI |
| `tcn_notime_filt_fine_cor_ndi` | coherence | TCN, no time | histogram match | NDI |
| `tcn_time_filt_fine_cor_ndi` | coherence | TCN + time | histogram match | NDI |
| `tcn_time_filt_fine_cor_logndi` | coherence | TCN + time | histogram match | log-NDI |

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess

BASE_DIR = Path('/scratch/yangyanchen/amatrice2025/')
GEOM_REF_DIR = Path('/scratch/yangyanchen/amatrice2025/')
CROPPED_DIR = BASE_DIR / 'cropped_paper_bbox'
EVENT_DATE = '20160824'
NEXT_DATE = '20160821_20160914'
LAT_MIN, LAT_MAX = 42.6, 42.7
LON_MIN, LON_MAX = 13.2, 13.4


def run_cmd(cmd: str, env=None):
    print(f"[RUN] {cmd}")
    merged_env = os.environ.copy()
    merged_env['PYTHONUNBUFFERED'] = '1'
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    return_code = proc.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with code {return_code}')


try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'CPU'
except Exception:
    GPU_AVAILABLE = False
    DEVICE_NAME = 'CPU'

print('BASE_DIR =', BASE_DIR)
print('CROPPED_DIR =', CROPPED_DIR)
print('EVENT_DATE =', EVENT_DATE)
print('NEXT_DATE =', NEXT_DATE)
print('Training device policy = CUDA if available else CPU')
print('Detected device =', DEVICE_NAME)

## 1. Crop interferogram pairs to the paper bounding box


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step crop "
    f"--base-dir {BASE_DIR} "
    f"--geom-reference-dir {GEOM_REF_DIR} "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-min {LAT_MIN} --lat-max {LAT_MAX} "
    f"--lon-min {LON_MIN} --lon-max {LON_MAX}"
)

## 2. Generate INT-derived auxiliary products

Prepares the `filt_fine.std` and related auxiliary products from INT files.


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step prepare_int_aux "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--aux-corr-win 5 --aux-phsig-win 5 "
    f"--aux-variance-win 5 --aux-variance-looks 3"
)

## 3. Inspect prepared products


In [ ]:
run_cmd(f"find {CROPPED_DIR} -maxdepth 1 -type f | sort")

## 4. Build datasets with histogram matching

Build RNN datasets for both `filt_fine.std` (phase-STD) and `filt_fine.cor` (coherence).
Histogram matching is applied to align the distribution of each time step to the median reference,
removing systematic acquisition-to-acquisition biases.


In [ ]:
# Build phase-STD dataset with histogram matching
run_cmd(
    f"python -m insar_pipeline.app --step build_dataset "
    f"--cropped-dir {CROPPED_DIR} --output-dir {CROPPED_DIR} "
    f"--event-date {EVENT_DATE} --observation-file filt_fine.std "
    f"--dataset-name dataset_rnn_filt_fine_std "
    f"--histogram-match --histogram-match-strategy median"
)

# Build coherence dataset with histogram matching
run_cmd(
    f"python -m insar_pipeline.app --step build_dataset "
    f"--cropped-dir {CROPPED_DIR} --output-dir {CROPPED_DIR} "
    f"--event-date {EVENT_DATE} --observation-file filt_fine.cor "
    f"--dataset-name dataset_rnn_filt_fine_cor "
    f"--histogram-match --histogram-match-strategy median"
)

run_cmd(
    f"python -m insar_pipeline.app --step validate_dataset "
    f"--output-dir {CROPPED_DIR}"
)

## 5. Pre/post NDI difference baselines

Simple normalized difference index between the last pre-event and first post-event observation.
No model is used — this is the traditional baseline.


In [ ]:
import numpy as np
from pathlib import Path
from insar_pipeline.scoring import calculate_difference

PREDICT_DIR = CROPPED_DIR / 'predict'
PREDICT_DIR.mkdir(parents=True, exist_ok=True)

def compute_prepost_ndi(dataset_dir: Path, observation_file: str, metric: str, output_name: str):
    """Load the score observation (post-event) and the last training image (pre-event),
    compute normalized difference, and save to PREDICT_DIR."""
    # Score observation = post-event
    obs_candidates = ['score_observation_std.npy', 'geninue_std.npy'] if metric == 'phase_std'         else ['score_observation.npy', 'geninue.npy']
    obs_file = next((dataset_dir / n for n in obs_candidates if (dataset_dir / n).exists()), None)
    if obs_file is None:
        print(f'WARNING: score observation not found in {dataset_dir}')
        return

    # Last time step in training series = pre-event
    ts_candidates = ['rnn_data_std.npy', 'data_std.npy'] if metric == 'phase_std'         else ['rnn_data.npy', 'data.npy']
    ts_file = next((dataset_dir / n for n in ts_candidates if (dataset_dir / n).exists()), None)
    if ts_file is None:
        print(f'WARNING: timeseries not found in {dataset_dir}')
        return

    post_event = np.load(obs_file).squeeze().astype(np.float32)
    timeseries = np.load(ts_file)
    pre_event = timeseries[:, :, -1].astype(np.float32)

    score = calculate_difference(pre_event, post_event, metric=metric)
    out_path = PREDICT_DIR / output_name
    np.save(out_path, score)
    print(f'Saved baseline NDI: {out_path}  shape={score.shape}')


compute_prepost_ndi(
    dataset_dir=CROPPED_DIR / 'dataset_rnn_filt_fine_std',
    observation_file='filt_fine.std',
    metric='phase_std',
    output_name='filt_fine_std_difference_score.npy',
)

compute_prepost_ndi(
    dataset_dir=CROPPED_DIR / 'dataset_rnn_filt_fine_cor',
    observation_file='filt_fine.cor',
    metric='coherence',
    output_name='filt_fine_cor_difference_score.npy',
)

## 6. TCN training and NDI scoring

Train a TCN model for each combination of metric × timestamp mode, then compute NDI scores.

- Uses **Huber loss** (robust to outlier pixels in SAR data)
- Uses **AdamW** optimizer with weight decay for better generalisation
- Uses 30 epochs with early stopping


In [ ]:
TCN_EXPERIMENTS = [
    {
        'observation_file': 'filt_fine.std',
        'metric': 'phase_std',
        'dataset_name': 'dataset_rnn_filt_fine_std',
        'timestamp_modes': ['notime', 'time'],
    },
    {
        'observation_file': 'filt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_filt_fine_cor',
        'timestamp_modes': ['notime', 'time'],
    },
]

for exp in TCN_EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        metric_tag = exp['metric']
        time_tag = timestamp_flag
        dataset_dir = CROPPED_DIR / exp['dataset_name']

        print('\n' + '=' * 80)
        print(f'TCN training: metric={metric_tag} timestamp={time_tag}')
        print('=' * 80)

        # Train
        run_cmd(
            f"python -m insar_pipeline.app --step train_predict "
            f"--dataset-dir {dataset_dir} "
            f"--output-dir {CROPPED_DIR} "
            f"--next-date {NEXT_DATE} "
            f"--timeseries-metric {metric_tag} "
            f"--ts-model tcn "
            f"--loss-type huber "
            f"--optimizer adamw --weight-decay 1e-4 "
            f"--epochs 30 "
            f"--rnn-hidden-dim 64 --rnn-num-layers 3 "
            f"{disable}"
        )

        # Score (NDI)
        score_filename = f"tcn_{time_tag}_{exp['observation_file'].replace('.','_')}_ndi_score.npy"
        run_cmd(
            f"python -m insar_pipeline.app --step score "
            f"--dataset-dir {dataset_dir} "
            f"--output-dir {CROPPED_DIR} "
            f"--timeseries-metric {metric_tag} "
            f"--ts-model tcn "
            f"--score-mode ndi "
            f"--score-filename {score_filename} "
            f"{disable}"
        )
        print(f'NDI score saved: predict/{score_filename}')

## 7. TCN log-NDI scoring

Reuse the predictions from Section 6 (no retraining needed).
log-NDI = log(observed / predicted) for phase_std, log(predicted / observed) for coherence.
A positive value indicates damage.


In [ ]:
for exp in TCN_EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        metric_tag = exp['metric']
        time_tag = timestamp_flag
        dataset_dir = CROPPED_DIR / exp['dataset_name']

        print('\n' + '-' * 80)
        print(f'TCN log-NDI scoring: metric={metric_tag} timestamp={time_tag}')

        score_filename = f"tcn_{time_tag}_{exp['observation_file'].replace('.','_')}_logndi_score.npy"
        run_cmd(
            f"python -m insar_pipeline.app --step score "
            f"--dataset-dir {dataset_dir} "
            f"--output-dir {CROPPED_DIR} "
            f"--timeseries-metric {metric_tag} "
            f"--ts-model tcn "
            f"--score-mode log_ndi "
            f"--score-filename {score_filename} "
            f"{disable}"
        )
        print(f'log-NDI score saved: predict/{score_filename}')

## 8. AUC evaluation

Load all score maps and compute AUC against the ground-truth damage mask.
`DAMAGE_MASK_FILE` should be a binary `.npy` array (1 = damaged, 0 = undamaged) spatially aligned with the score maps.


In [ ]:
import numpy as np
from pathlib import Path

DAMAGE_MASK_FILE = CROPPED_DIR / 'damage_mask.npy'  # <-- set your mask path here
PREDICT_DIR = CROPPED_DIR / 'predict'

# Score-file label mapping  (filename → display label)
SCORE_LABEL_MAP = {
    'filt_fine_std_difference_score.npy':             'filt_fine_std_difference',
    'filt_fine_cor_difference_score.npy':             'filt_fine_cor_difference',
    'tcn_notime_filt_fine_std_ndi_score.npy':         'tcn_notime_filt_fine_std_ndi',
    'tcn_time_filt_fine_std_ndi_score.npy':           'tcn_time_filt_fine_std_ndi',
    'tcn_time_filt_fine_std_logndi_score.npy':        'tcn_time_filt_fine_std_logndi',
    'tcn_notime_filt_fine_cor_ndi_score.npy':         'tcn_notime_filt_fine_cor_ndi',
    'tcn_time_filt_fine_cor_ndi_score.npy':           'tcn_time_filt_fine_cor_ndi',
    'tcn_time_filt_fine_cor_logndi_score.npy':        'tcn_time_filt_fine_cor_logndi',
}

if not DAMAGE_MASK_FILE.exists():
    print(f'WARNING: damage mask not found at {DAMAGE_MASK_FILE}')
    print('Set DAMAGE_MASK_FILE above and re-run this cell to compute AUC values.')
else:
    from sklearn.metrics import roc_auc_score, f1_score, roc_curve

    damage_mask = np.load(DAMAGE_MASK_FILE).astype(np.float32)
    if damage_mask.ndim == 3:
        damage_mask = damage_mask.squeeze()
    valid = ~np.isnan(damage_mask)

    header = f"{'Run name':<45} {'AUC':<10} {'Best F1':<10} {'Threshold':<12} {'FPR':<10}"
    sep = '-' * 90
    print(sep)
    print(header)
    print(sep)

    for score_file, label in SCORE_LABEL_MAP.items():
        score_path = PREDICT_DIR / score_file
        if not score_path.exists():
            print(f'  {label:<43} -- score file not found')
            continue
        score_map = np.load(score_path).astype(np.float32)
        if score_map.ndim == 3:
            score_map = score_map.squeeze()
        combined_valid = valid & ~np.isnan(score_map)
        y_true_sub = damage_mask[combined_valid].astype(int)
        y_score_sub = score_map[combined_valid]
        if len(np.unique(y_true_sub)) < 2:
            print(f'  {label:<43} -- only one class in valid mask, skipping')
            continue
        auc = roc_auc_score(y_true_sub, y_score_sub)
        fpr_arr, tpr_arr, thresholds = roc_curve(y_true_sub, y_score_sub)
        f1_scores = [
            f1_score(y_true_sub, (y_score_sub >= thr).astype(int), zero_division=0)
            for thr in thresholds
        ]
        best_idx = int(np.argmax(f1_scores))
        print(f'{label:<45} {auc:<10.5f} {f1_scores[best_idx]:<10.5f} '
              f'{thresholds[best_idx]:<12.5f} {fpr_arr[best_idx]:<10.5f}')
    print(sep)